In [ ]:
#1

# Install PySpark
!pip install pyspark -q

# Download the official MySQL JDBC Driver (Required for TiDB)
!wget https://repo1.maven.org/maven2/mysql/mysql-connector-java/8.0.28/mysql-connector-java-8.0.28.jar -q

In [ ]:
#2
# --- Updated Cell #2 (Scala 2.13 Version) ---
import os
from pyspark.sql import SparkSession

# We change 'spark-xml_2.12' to 'spark-xml_2.13' to match Spark 4.0.2
spark = SparkSession.builder \
    .appName("FFIEC_Big_Bertha") \
    .config("spark.jars", "mysql-connector-java-8.0.28.jar") \
    .config("spark.jars.packages", "com.databricks:spark-xml_2.13:0.17.0") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

print("🚀 Spark 4.0.2 is running with the correct XML (2.13) package!")

print("🚀 Spark is running!")
print("Spark Version:", spark.version)

In [ ]:
#3a

# 1. Clear out the broken Ubuntu Chromium packages
!apt-ge
t purge chromium-browser chromium-chromedriver -y -q
!apt-get autoremove -y -q

# 2. Install the official Google Chrome Stable package
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'
!apt-get update -q
!apt-get install -y google-chrome-stable -q

# 3. Install Selenium and the automatic driver manager
!pip install selenium webdriver-manager -q

In [ ]:
#3b

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import os
import zipfile

# Modern Headless Setup
chrome_options = Options()
chrome_options.add_argument('--headless=new') # The new way to call headless Chrome
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

TEMP_DIR = "/content/ffiec_temp"
os.makedirs(TEMP_DIR, exist_ok=True)

# Route downloads to our temp folder
prefs = {'download.default_directory' : TEMP_DIR}
chrome_options.add_experimental_option('prefs', prefs)

print("Initializing Chrome...")
# webdriver-manager automatically downloads the exact driver needed for our Chrome version
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)
wait = WebDriverWait(driver, 20)

mvp_periods = ["12/31/2019", "12/31/2020", "12/31/2021", "12/31/2022", "12/31/2023"]

for period in mvp_periods:
    print(f"Loading FFIEC Web Interface for {period}...")
    driver.get("https://cdr.ffiec.gov/public/pws/downloadbulkdata.aspx")

    # 1. Select "Call Reports -- Single Period"
    product_dropdown = wait.until(EC.presence_of_element_located((By.ID, "ListBox1")))
    Select(product_dropdown).select_by_visible_text("Call Reports -- Single Period")
    time.sleep(3)

    # 2. Select the specific Date
    dates_dropdown = wait.until(EC.presence_of_element_located((By.ID, "DatesDropDownList")))
    try:
        Select(dates_dropdown).select_by_visible_text(period)
    except Exception as e:
        print(f"  -> Could not find {period} in dropdown. Skipping.")
        continue

    # CRITICAL: Wait for the page to "blink" (refresh) after date selection
    time.sleep(4)

    # --- FIXED: Step 2.5: Select the XBRL Radio Button (Using your working ID) ---
    print("Selecting XBRL format...")
    try:
        # Using the ID 'XBRLRadiobutton' from your update_engine.py
        xbrl_radio = wait.until(EC.element_to_be_clickable((By.ID, "XBRLRadiobutton")))
        driver.execute_script("arguments[0].click();", xbrl_radio)
        print("  -> XBRL format selected successfully.")
    except Exception as e:
        print(f"  -> Critical Error: Could not find the XBRL radio button. {e}")
        driver.save_screenshot("debug_error.png") # Handy for seeing what the robot sees
        continue

    # Wait for the server to react to the radio button click
    time.sleep(3)

    # 3. Click Download
    print("Triggering download... (This is a 50MB file, give it a moment)")
    download_btn = wait.until(EC.element_to_be_clickable((By.ID, "Download_0")))
    driver.execute_script("arguments[0].click();", download_btn)

    # 4. Wait for the zip file to finish downloading
    timeout = 60
    start_time = time.time()
    zip_path = None

    while time.time() - start_time < timeout:
        files = os.listdir(TEMP_DIR)
        crdownloads = [f for f in files if f.endswith(".crdownload")]
        zips = [f for f in files if f.endswith(".zip")]

        if zips and not crdownloads:
            zip_path = os.path.join(TEMP_DIR, zips[0])
            break
        time.sleep(2)

    if not zip_path:
        print(f"  -> Download failed or timed out for {period}")
        continue

    # Re-run your downloader logic, but REPLACE Step 5 with this:

    # 5. Extract EVERYTHING and DO NOT DELETE the ZIP yet
    print(f"Extracting ALL data for {period}...")
    period_folder = os.path.join(TEMP_DIR, period.replace('/', ''))
    os.makedirs(period_folder, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(period_folder)

    # WE ARE NOT RUNNING os.remove(zip_path) THIS TIME
    print(f"  -> Successfully extracted all files for {period}!")

driver.quit()
print("✅ All downloads complete! Ready for Spark.")

In [ ]:
!pip install mysql-connector-python -q

In [ ]:
!pip install mysql-connector-python -q

In [ ]:
import mysql.connector

try:
    # We connect without a database name first to create it
    conn = mysql.connector.connect(
        host="gateway01.us-east-1.prod.aws.tidbcloud.com",
        port=4000,
        user="4Gn7q79SCLgZvN4.root",
        password="UHOwMzLZriA76JWj",
        ssl_verify_cert=True
    )
    cursor = conn.cursor()
    cursor.execute("CREATE DATABASE IF NOT EXISTS ffiec_data;")
    print("✅ Database 'ffiec_data' created or already exists.")
    cursor.close()
    conn.close()
except Exception as e:
    print(f"❌ Connection Error: {e}")

# Update the Spark URL for the next step
DB_NAME = "ffiec_data"
DB_URL = f"jdbc:mysql://gateway01.us-east-1.prod.aws.tidbcloud.com:4000/{DB_NAME}?sslMode=VERIFY_IDENTITY"

In [ ]:
# --- Updated Cell #4 (Push to Correct DB) ---
import glob
import os
from pyspark.sql.functions import input_file_name, regexp_extract

# 1. Find all POR files
txt_files = glob.glob(os.path.join(TEMP_DIR, "**/*POR*.txt"), recursive=True)

if len(txt_files) > 0:
    df = spark.read.option("header", "true").option("delimiter", "\t").csv(txt_files)

    # Clean columns
    for col_name in df.columns:
        if "IDRSSD" in col_name.upper():
            df = df.withColumnRenamed(col_name, "idrssd")
        if "FINANCIAL INSTITUTION NAME" in col_name.upper():
            df = df.withColumnRenamed(col_name, "bank_name")

    df_clean = df.withColumn("source_folder", regexp_extract(input_file_name(), r"(\d{8})", 1)) \
                 .select("idrssd", "bank_name", "source_folder")

    # 2. Push to TiDB (Now using /ffiec_data)
    print(f"Pushing {df_clean.count()} bank records to TiDB...")
    df_clean.write \
        .format("jdbc") \
        .option("driver", "com.mysql.cj.jdbc.Driver") \
        .option("url", DB_URL + "&rewriteBatchedStatements=true") \
        .option("dbtable", "call_reports_por") \
        .option("user", DB_USER) \
        .option("password", DB_PASSWORD) \
        .option("isolationLevel", "NONE") \
        .option("batchsize", "10000") \
        .mode("overwrite") \
        .save()

    print("🎉 Success! The bank directory is now in your TiDB 'ffiec_data' database.")

In [ ]:
!rm -rf /content/ffiec_temp/*


In [ ]:
# --- Stage #5: Pure Python Multiprocessing Engine ---
import glob
import os
import xml.etree.ElementTree as ET
import mysql.connector
from concurrent.futures import ThreadPoolExecutor
import re

# --- CONFIGURATION ---
TEST_MODE = False      # True = 10 files, False = All files
TARGET_TABLE = "call_reports_financials"
TEMP_DIR = "/content/ffiec_temp"

# TiDB Database config
db_config = {
    "host": "gateway01.us-east-1.prod.aws.tidbcloud.com",
    "port": 4000,
    "user": "4Gn7q79SCLgZvN4.root",
    "password": "UHOwMzLZriA76JWj",
    "database": "ffiec_data",
    "ssl_verify_cert": True
}

print("1. Gathering XML files...")
all_xml_files = glob.glob(os.path.join(TEMP_DIR, "**/*.xml"), recursive=True)
xml_files = all_xml_files[:10] if TEST_MODE else all_xml_files

if not xml_files:
    print("❌ No XML files found!")
else:
    print("2. Setting up TiDB Table...")
    # Create the table using a direct connection
    conn = mysql.connector.connect(**db_config)
    cursor = conn.cursor()
    cursor.execute(f"""
    CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
        id INT AUTO_INCREMENT PRIMARY KEY,
        idrssd INT,
        report_date VARCHAR(10),
        concept_reference VARCHAR(100),
        value TEXT,
        unit_ref VARCHAR(50),
        context_ref VARCHAR(100)
    )
    """)
    conn.commit()
    cursor.close()
    conn.close()

    # 3. The Worker Function (Adapted from your update_engine.py)
    def process_and_push(filepath):
        try:
            # Extract date from folder path
            match = re.search(r'(\d{8})', filepath)
            report_date = match.group(1) if match else "Unknown"

            tree = ET.parse(filepath)
            root = tree.getroot()

            idrssd = None
            for elem in root.iter():
                if elem.tag.endswith('identifier') and elem.text:
                    try:
                        idrssd = int(elem.text)
                        break
                    except ValueError:
                        pass

            if not idrssd: return 0

            rows_to_insert = []
            for child in root:
                if 'contextRef' in child.attrib:
                    concept_ref = child.tag.split('}')[-1]
                    value = child.text.strip() if child.text else None
                    unit_ref = child.attrib.get('unitRef')
                    context_ref = child.attrib.get('contextRef')

                    if value is not None:
                        rows_to_insert.append((
                            idrssd, report_date, concept_ref, value, unit_ref, context_ref
                        ))

            # Push to database using executemany for speed
            if rows_to_insert:
                local_conn = mysql.connector.connect(**db_config)
                local_cursor = local_conn.cursor()
                query = f"""
                    INSERT INTO {TARGET_TABLE}
                    (idrssd, report_date, concept_reference, value, unit_ref, context_ref)
                    VALUES (%s, %s, %s, %s, %s, %s)
                """
                # We chunk it into batches of 5000 to keep the cloud connection stable
                chunk_size = 5000
                for i in range(0, len(rows_to_insert), chunk_size):
                    local_cursor.executemany(query, rows_to_insert[i:i+chunk_size])

                local_conn.commit()
                local_cursor.close()
                local_conn.close()

            return len(rows_to_insert)
        except Exception as e:
            return 0

    print(f"3. Processing {len(xml_files)} files in parallel...")
    total_inserted = 0

    # Use ThreadPool to process multiple files at once over network connections
    with ThreadPoolExecutor(max_workers=8) as executor:
        for rows_count in executor.map(process_and_push, xml_files):
            total_inserted += rows_count

    print(f"🏁 SUCCESS! Uploaded {total_inserted} financial data rows to TiDB.")

In [ ]:
from sqlalchemy import create_engine, text

# Pointing explicitly to /ffiec_data

print("Connecting to TiDB (ffiec_data)...")
# AUTOCOMMIT is required to build/drop indexes via SQLAlchemy
engine = create_engine(DB_URL, isolation_level="AUTOCOMMIT")

indexes = [
    ("idx_rssd", "idrssd"),
    ("idx_date", "report_date"),
    ("idx_rssd_date", "idrssd, report_date")
]

with engine.connect() as conn:
    print("\n--- PHASE 1: DROPPING OLD INDEXES ---")
    for idx_name, _ in indexes:
        try:
            print(f"Attempting to drop {idx_name}...")
            conn.execute(text(f"DROP INDEX {idx_name} ON ffiec_data.call_reports_financials;"))
            print(f" 🗑️ Dropped {idx_name}.")
        except Exception as e:
            # It's totally fine if it fails here (it just means the index didn't exist yet)
            print(f" ⏩ Skipped dropping {idx_name} (likely didn't exist).")

    print("\n--- PHASE 2: CREATING FRESH INDEXES ---")
    print("WARNING: This may take 15-30 minutes for 25 million rows.")
    print("Go grab a coffee and leave this script running until you see the success message!\n")

    for idx_name, columns in indexes:
        try:
            print(f"Building {idx_name} on columns: ({columns})...")
            conn.execute(text(f"CREATE INDEX {idx_name} ON ffiec_data.call_reports_financials({columns});"))
            print(f" ✅ {idx_name} built successfully!")
        except Exception as e:
            print(f" ❌ Error building {idx_name}: {e}")

print("\n🎉 ALL DONE! Your database is completely refreshed, optimized, and ready for Streamlit!")